# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [2]:
#Import your modules here
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.credit_clustering import (
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,
)


PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "outputs" / "saved_models" / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model artifact not found at {MODEL_PATH}. "
        "Copy nonfinancial_credit_kmeans_k5.joblib into saved_models/ or update MODEL_PATH."
    )

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("Exists:", INPUT_PATH.exists())

artifact = joblib.load(MODEL_PATH)

print("Loaded artifact:", artifact.get("model_name"))
print("Segment:", artifact.get("segment"))
print("Training rows:", artifact.get("training_rows"))
print("Artifact keys:", list(artifact.keys()))

pipe = artifact["pipeline"]
feature_cols = artifact["feature_cols"]
cluster_labels = artifact["cluster_labels"]
risk_rank = artifact["risk_rank"]
winsor_caps = artifact.get("winsor_caps")

print("Features used by model:", feature_cols)
print("Cluster labels:", cluster_labels)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
INPUT_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs
Exists: True
Loaded artifact: nonfinancial_credit_scorecard_kmeans_k5_v3
Segment: Non-financial
Training rows: 21265
Artifact keys: ['model_name', 'version', 'segment', 'pipeline', 'feature_cols', 'cluster_labels', 'risk_rank', 'scorecard_thresholds', 'scorecard_domain_weights', 'min_non_null_features', 'scorecard_cluster_features', 'scorecard_component_features', 'winsor_caps', 'training_rows', 'cluster_profile_ranked', 'metrics_df', 'notes']
Features used by model: ['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']
Cluster labels: {1: '1 - Low risk / investment-grade-like', 2: '2 - Moderate risk / lower-investment-grade-like', 0: '3 - Elevated risk / leveraged', 3: '4 - High

## 2. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025 = pd.read_csv(template_path)
manual_input_2025

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Manual 2025 Company,2025,USD,Manufacturing / Industrials,Non-financial,48585294.12,29721275.23,10037745.82,34804.65,2208235.29,7794705.88,9082352.94,12722352.94,20638823.53,7910588.24,1478235.29,949411.76,3558235.29,588235.29,1664705.88,597647.06,2713000,NaN


In [5]:
FX_TO_MODEL_CURRENCY = 1.0

scored_manual = score_companies(
    manual_input_2025,
    artifact=artifact,
    segment="Non-financial",
    temperature=1.0,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

summary_cols_with_outlook = [
    "company_name",
    "fiscal_year",
    "assigned_cluster",
    "cluster_label",
    "risk_rank",
    "scorecard_credit_score",
    "cluster_affinity",
    "near_default_affinity",
    "distance_to_assigned_cluster",
    "upper_bucket_cluster",
    "upper_bucket_label",
    "distance_to_upper_bucket",
    "lower_bucket_cluster",
    "lower_bucket_label",
    "distance_to_lower_bucket",
    "outlook_flag",
    "outlook_reason",
    "feature_coverage_pct",
    "warning_flags",
]

missing_cols = [
    col for col in summary_cols_with_outlook
    if col not in scored_manual_with_outlook.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)
    print("Available columns:", list(scored_manual_with_outlook.columns))

existing_summary_cols_with_outlook = [
    col for col in summary_cols_with_outlook
    if col in scored_manual_with_outlook.columns
]

scored_manual_with_outlook[existing_summary_cols_with_outlook]


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,scorecard_credit_score,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags
0,Manual 2025 Company,2025,2,2 - Moderate risk / lower-investment-grade-like,2,30.006705,0.364935,0.113406,0.416886,1,1 - Low risk / investment-grade-like,0.929087,0,3 - Elevated risk / leveraged,1.061683,Neutral,Company remains materially closer to its assig...,1.0,"current_ratio_below_1, quick_ratio_below_0_5"


In [6]:
# Prefer the ratio list saved by Notebook 02's patched artifact.
# Keep the fallback list so this notebook remains usable with older artifacts.
ratio_cols = artifact.get("ratio_cols", [
    "log_assets",
    "liabilities_to_assets",
    "equity_to_assets",
    "cash_to_assets",
    "revenue_to_assets",
    "net_income_to_assets",
    "cfo_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "current_ratio",
    "quick_ratio",
    "gross_margin",
    "operating_margin",
    "interest_coverage",
    "cfo_to_liabilities",
    "fcf_to_debt",
    "capex_to_revenue",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
])

additional_diagnostic_cols = [
    "scorecard_credit_score",
    "total_debt",
    "net_debt",
    "ebitda",
    "debt_service_risk",
    "debt_service_risk_legacy",
    "ebitda_margin_risk",
    "debt_to_ebitda_risk",
    "net_debt_to_ebitda_risk",
    "ebitda_coverage_risk",
    "negative_ebitda_flag",
]

manual_diagnostic_cols = [
    c for c in additional_diagnostic_cols + ratio_cols
    if c in scored_manual.columns
]

scored_manual[["company_name"] + manual_diagnostic_cols].T


,0
company_name,Manual 2025 Company
scorecard_credit_score,30.006705
total_debt,9388823.53
net_debt,9354018.88
ebitda,4377705.88
debt_service_risk,0.046592
debt_service_risk_legacy,0.06437
ebitda_margin_risk,0.549482
debt_to_ebitda_risk,0.036173
net_debt_to_ebitda_risk,0.181926


## 3. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [7]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,17.2074,22.9073,-5.6999,below_cluster_median
liabilities_to_assets,0.6944,0.6673,0.0271,above_cluster_median
debt_to_assets,0.3159,0.2948,0.0211,above_cluster_median
equity_to_assets,0.3056,0.3038,0.0018,above_cluster_median
current_ratio,0.7890,0.9989,-0.2099,below_cluster_median
quick_ratio,0.1763,0.3407,-0.1644,below_cluster_median
cash_to_assets,0.0012,0.0287,-0.0276,below_cluster_median
net_income_to_assets,0.0319,0.0336,-0.0017,below_cluster_median
cfo_to_assets,0.1197,0.0809,0.0388,above_cluster_median


## 4. Scenario analysis

This section shows how a company migrates across clusters under stress cases

In [8]:
scenario_input = make_scenarios(manual_input_2025.iloc[0].to_dict())

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "interest_expense",
    "operating_income",
    "depreciation_amortization",
    "ebitda",
]

scenario_input[[c for c in scenario_input_cols if c in scenario_input.columns]]


,scenario,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,interest_expense,operating_income,depreciation_amortization,ebitda
0,base,29721275.23,2.063882e+07,9082352.940,34804.650,4.858529e+07,949411.760,3.558235e+06,7.910588e+06,1.478235e+06,597647.06,1664705.880,2713000,NaN
1,revenue_down_15pct,29721275.23,2.063882e+07,9082352.940,34804.650,4.129750e+07,664588.232,2.668676e+06,7.910588e+06,1.478235e+06,597647.06,1165294.116,2713000,NaN
2,debt_up_25pct,29721275.23,2.261647e+07,9082352.940,0.000,4.858529e+07,949411.760,3.558235e+06,9.888235e+06,1.478235e+06,597647.06,1664705.880,2713000,NaN
3,cash_burn_case,29721275.23,2.063882e+07,9082352.940,17402.325,4.858529e+07,-949411.760,-3.558235e+06,7.910588e+06,1.478235e+06,597647.06,1664705.880,2713000,NaN
4,near_default_stress,29721275.23,3.269340e+07,-2972127.523,34804.650,4.858529e+07,949411.760,3.558235e+06,2.229096e+07,4.458191e+06,1000000.00,239058.824,2713000,NaN


In [9]:
scored_scenarios = score_companies(
    scenario_input,
    artifact=artifact,
    segment="Non-financial",
    temperature=1.0,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
)

scenario_summary_cols = [
    "scenario",
    "assigned_cluster",
    "cluster_label",
    "risk_rank",
    "scorecard_credit_score",
    "cluster_affinity",
    "near_default_affinity",
    "liabilities_to_assets",
    "debt_to_assets",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
    "warning_flags",
]

existing_scenario_summary_cols = [
    c for c in scenario_summary_cols
    if c in scored_scenarios.columns
]

scored_scenarios[existing_scenario_summary_cols]


,scenario,assigned_cluster,cluster_label,risk_rank,scorecard_credit_score,cluster_affinity,near_default_affinity,liabilities_to_assets,debt_to_assets,equity_to_assets,cfo_to_assets,current_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk,warning_flags
0,base,2,2 - Moderate risk / lower-investment-grade-like,2,30.006705,0.364935,0.113406,0.694412,0.315896,0.305584,0.11972,0.788985,2.785433,0.316334,4377705.880,0.090104,2.144690,2.136740,7.324902,0.046592,"current_ratio_below_1, quick_ratio_below_0_5"
1,revenue_down_15pct,2,2 - Moderate risk / lower-investment-grade-like,2,33.896620,0.369850,0.112534,0.694412,0.315896,0.305584,0.08979,0.788985,1.949803,0.221587,3878294.116,0.093911,2.420864,2.411890,6.489272,0.210088,"current_ratio_below_1, quick_ratio_below_0_5"
2,debt_up_25pct,2,2 - Moderate risk / lower-investment-grade-like,2,32.610411,0.365338,0.114768,0.760952,0.382435,0.305584,0.11972,0.788985,2.785433,0.261295,4377705.880,0.090104,2.596445,2.596445,7.324902,0.074827,"current_ratio_below_1, quick_ratio_below_0_5"
3,cash_burn_case,0,3 - Elevated risk / leveraged,3,63.339858,0.289285,0.175139,0.694412,0.315896,0.305584,-0.11972,0.788985,2.785433,-0.441639,4377705.880,0.090104,2.144690,2.140715,7.324902,0.296592,"current_ratio_below_1, quick_ratio_below_0_5, ..."
4,near_default_stress,4,5 - Distressed / near-default proxy,5,62.955405,0.254550,0.254550,1.100000,0.900000,-0.100000,0.11972,0.788985,0.239059,0.111032,2952058.824,0.060760,9.061184,9.049394,2.952059,0.701845,"liabilities_exceed_assets, negative_equity, hi..."


## 5. Save report outputs to files


In [10]:
# ---------------------------------------------------------------------
# 1. Main score summary with outlook
# ---------------------------------------------------------------------
score_summary = scored_manual_with_outlook[existing_summary_cols_with_outlook].copy()


# ---------------------------------------------------------------------
# 2. Company ratio and diagnostic snapshot
# ---------------------------------------------------------------------
company_ratio_cols = ["company_name"] + [
    c for c in additional_diagnostic_cols + ratio_cols
    if c in scored_manual.columns
]

company_ratios = scored_manual[company_ratio_cols].copy()


# ---------------------------------------------------------------------
# 3. Cluster comparison table
# comparison_to_cluster has metrics as index, so reset it before saving.
# ---------------------------------------------------------------------
cluster_comparison = comparison_to_cluster.reset_index().copy()

if "index" in cluster_comparison.columns:
    cluster_comparison = cluster_comparison.rename(columns={"index": "metric"})


# ---------------------------------------------------------------------
# 4. Create one flat CSV-friendly output
# ---------------------------------------------------------------------
# Main output: summary + ratios/diagnostics in one row.
scored_file = score_summary.merge(
    company_ratios,
    on="company_name",
    how="left",
)


# ---------------------------------------------------------------------
# 5. Scenario analysis outputs
# ---------------------------------------------------------------------

if "scenario" not in scored_scenarios.columns:
    scored_scenarios["scenario"] = scenario_input["scenario"].values

# Scenario input snapshot
scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "interest_expense",
    "operating_income",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    c for c in scenario_input_cols
    if c in scenario_input.columns
]

scenario_input_snapshot = scenario_input[existing_scenario_input_cols].copy()


# Scenario scoring summary
existing_scenario_summary_cols = [
    c for c in scenario_summary_cols
    if c in scored_scenarios.columns
]

scenario_score_summary = scored_scenarios[existing_scenario_summary_cols].copy()


# Scenario ratios and diagnostics
scenario_ratio_cols = ["scenario"] + [
    c for c in additional_diagnostic_cols + ratio_cols
    if c in scored_scenarios.columns
]

scenario_ratios = scored_scenarios[scenario_ratio_cols].copy()


# Flat CSV-friendly scenario output
scenario_file = scenario_score_summary.merge(
    scenario_ratios,
    on="scenario",
    how="left",
)

# Avoid duplicate columns after merge if a metric appears in both summary and ratio sections.
duplicate_suffix_cols = [c for c in scenario_file.columns if c.endswith("_y")]
scenario_file = scenario_file.drop(columns=duplicate_suffix_cols)
scenario_file.columns = [c[:-2] if c.endswith("_x") else c for c in scenario_file.columns]


# ---------------------------------------------------------------------
# 6. Save base and scenario outputs
# ---------------------------------------------------------------------
output_file = OUTPUT_PATH / "manual_2025_score_result.csv"
OUTPUT_XLSX = OUTPUT_PATH / "manual_2025_score_report.xlsx"

OUTPUT_SCENARIO_CSV = OUTPUT_PATH / "manual_2025_scenario_analysis.csv"
OUTPUT_SCENARIO_XLSX = OUTPUT_PATH / "manual_2025_scenario_analysis.xlsx"

scored_file.to_csv(output_file, index=False)
scenario_file.to_csv(OUTPUT_SCENARIO_CSV, index=False)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    score_summary.to_excel(writer, sheet_name="score_summary", index=False)
    company_ratios.to_excel(writer, sheet_name="company_ratios", index=False)
    cluster_comparison.to_excel(writer, sheet_name="cluster_comparison", index=False)
    scenario_input_snapshot.to_excel(writer, sheet_name="scenario_inputs", index=False)
    scenario_score_summary.to_excel(writer, sheet_name="scenario_scores", index=False)
    scenario_ratios.to_excel(writer, sheet_name="scenario_ratios", index=False)
    scenario_file.to_excel(writer, sheet_name="scenario_full_output", index=False)

with pd.ExcelWriter(OUTPUT_SCENARIO_XLSX, engine="openpyxl") as writer:
    scenario_input_snapshot.to_excel(writer, sheet_name="scenario_inputs", index=False)
    scenario_score_summary.to_excel(writer, sheet_name="scenario_scores", index=False)
    scenario_ratios.to_excel(writer, sheet_name="scenario_ratios", index=False)
    scenario_file.to_excel(writer, sheet_name="scenario_full_output", index=False)

print("Scores saved to:", output_file)
print("Full Excel score report saved to:", OUTPUT_XLSX)
print("Scenario CSV saved to:", OUTPUT_SCENARIO_CSV)
print("Scenario Excel report saved to:", OUTPUT_SCENARIO_XLSX)

display(scored_file)
display(scenario_file)


Scores saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\manual_2025_score_result.csv
Full Excel score report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\manual_2025_score_report.xlsx
Scenario CSV saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\manual_2025_scenario_analysis.csv
Scenario Excel report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\manual_2025_scenario_analysis.xlsx


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,scorecard_credit_score_x,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags,scorecard_credit_score_y,total_debt,net_debt,ebitda,debt_service_risk,debt_service_risk_legacy,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,log_assets,liabilities_to_assets,equity_to_assets,cash_to_assets,revenue_to_assets,net_income_to_assets,cfo_to_assets,debt_to_assets,debt_to_equity,current_ratio,quick_ratio,gross_margin,operating_margin,interest_coverage,cfo_to_liabilities,fcf_to_debt,capex_to_revenue,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage
0,Manual 2025 Company,2025,2,2 - Moderate risk / lower-investment-grade-like,2,30.006705,0.364935,0.113406,0.416886,1,1 - Low risk / investment-grade-like,0.929087,0,3 - Elevated risk / leveraged,1.061683,Neutral,Company remains materially closer to its assig...,1.0,"current_ratio_below_1, quick_ratio_below_0_5",30.006705,9388823.53,9354018.88,4377705.88,0.046592,0.06437,0.549482,0.036173,0.181926,0.0,0.0,17.207374,0.694412,0.305584,0.001171,1.634697,0.031944,0.11972,0.315896,NaN,0.788985,0.176307,NaN,0.034264,2.785433,0.172405,0.316334,0.012107,0.090104,2.14469,2.13674,7.324902


,scenario,assigned_cluster,cluster_label,risk_rank,scorecard_credit_score,cluster_affinity,near_default_affinity,liabilities_to_assets,debt_to_assets,equity_to_assets,cfo_to_assets,current_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk,warning_flags,total_debt,net_debt,debt_service_risk_legacy,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,log_assets,cash_to_assets,revenue_to_assets,net_income_to_assets,debt_to_equity,quick_ratio,gross_margin,operating_margin,cfo_to_liabilities,capex_to_revenue
0,base,2,2 - Moderate risk / lower-investment-grade-like,2,30.006705,0.364935,0.113406,0.694412,0.315896,0.305584,0.11972,0.788985,2.785433,0.316334,4377705.880,0.090104,2.144690,2.136740,7.324902,0.046592,"current_ratio_below_1, quick_ratio_below_0_5",9.388824e+06,9.354019e+06,0.064370,0.549482,0.036173,0.181926,0.000000,0.0,17.207374,0.001171,1.634697,0.031944,NaN,0.176307,NaN,0.034264,0.172405,0.012107
1,revenue_down_15pct,2,2 - Moderate risk / lower-investment-grade-like,2,33.896620,0.369850,0.112534,0.694412,0.315896,0.305584,0.08979,0.788985,1.949803,0.221587,3878294.116,0.093911,2.420864,2.411890,6.489272,0.210088,"current_ratio_below_1, quick_ratio_below_0_5",9.388824e+06,9.354019e+06,0.315059,0.530444,0.105216,0.260540,0.000000,0.0,17.207374,0.001171,1.389493,0.022361,NaN,0.176307,NaN,0.028217,0.129304,0.014244
2,debt_up_25pct,2,2 - Moderate risk / lower-investment-grade-like,2,32.610411,0.365338,0.114768,0.760952,0.382435,0.305584,0.11972,0.788985,2.785433,0.261295,4377705.880,0.090104,2.596445,2.596445,7.324902,0.074827,"current_ratio_below_1, quick_ratio_below_0_5",1.136647e+07,1.136647e+07,0.064370,0.549482,0.149111,0.313270,0.000000,0.0,17.207374,0.000000,1.634697,0.031944,NaN,0.173571,NaN,0.034264,0.157329,0.012107
3,cash_burn_case,0,3 - Elevated risk / leveraged,3,63.339858,0.289285,0.175139,0.694412,0.315896,0.305584,-0.11972,0.788985,2.785433,-0.441639,4377705.880,0.090104,2.144690,2.140715,7.324902,0.296592,"current_ratio_below_1, quick_ratio_below_0_5, ...",9.388824e+06,9.371421e+06,0.464370,0.549482,0.036173,0.183061,0.000000,0.0,17.207374,0.000586,1.634697,-0.031944,NaN,0.174939,NaN,0.034264,-0.172405,0.012107
4,near_default_stress,4,5 - Distressed / near-default proxy,5,62.955405,0.254550,0.254550,1.100000,0.900000,-0.100000,0.11972,0.788985,0.239059,0.111032,2952058.824,0.060760,9.061184,9.049394,2.952059,0.701845,"liabilities_exceed_assets, negative_equity, hi...",2.674915e+07,2.671434e+07,0.662349,0.696198,1.000000,1.000000,0.419176,0.0,17.207374,0.001171,1.634697,0.031944,NaN,0.176307,NaN,0.004920,0.108836,0.012107


## 6. Optional: inspect saved artifact internals

In [11]:
nonfin_artifact = artifact

print("Features used by model:")
print(nonfin_artifact["feature_cols"])

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))

Features used by model:
['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']

Cluster labels:
{1: '1 - Low risk / investment-grade-like', 2: '2 - Moderate risk / lower-investment-grade-like', 0: '3 - Elevated risk / leveraged', 3: '4 - High risk / speculative', 4: '5 - Distressed / near-default proxy'}

Cluster sizes:


None


Cluster profile:


None


Cluster profile ranked:


,financial_flag,cluster,issuer_years,issuers,median_scorecard_credit_score,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_liabilities_risk,median_debt_load_risk,median_equity_buffer_risk,median_cash_buffer_risk,median_current_liquidity_risk,median_quick_liquidity_risk,median_profitability_risk,median_cashflow_risk,median_coverage_risk,median_fcf_risk,median_negative_equity_flag,median_liabilities_exceed_assets_flag,median_log_assets,median_small_company_flag,median_mid_company_flag,median_large_company_flag,median_liabilities_to_assets,median_debt_to_assets,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_current_ratio,median_quick_ratio,median_revenue_to_assets,median_interest_coverage,median_cfo_to_liabilities,median_fcf_to_debt,median_operating_margin,median_gross_margin,rating_style_rank,rating_style_label
1,Non-financial,1,6611,1809,10.406647,0.068963,0.068382,0.000000,0.000000,0.000000,0.0,0.006684,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,21.288508,0.0,0.0,1.0,0.453676,0.234827,0.484361,0.115473,0.051673,0.095050,2.352138,1.263870,0.606353,6.634322,0.200625,0.258875,0.113507,0.372474,1,1 - Low risk / investment-grade-like
2,Non-financial,2,4115,1167,32.521368,0.262992,0.750781,0.164049,0.000000,0.126039,0.0,0.395112,0.074701,0.240504,0.791981,0.800868,0.879065,0.164049,0.000000,0.0,0.000000,0.0,0.0,22.907293,0.0,0.0,1.0,0.667311,0.294821,0.303799,0.028722,0.033595,0.080923,0.998915,0.340702,0.478907,3.195531,0.123600,0.172850,0.132343,0.322081,2,2 - Moderate risk / lower-investment-grade-like
0,Non-financial,0,2918,1357,51.499150,0.270081,0.573432,0.973705,0.534669,0.823495,0.0,0.336748,0.021737,0.180379,0.454847,0.622640,0.629066,0.973705,0.534669,1.0,0.625716,0.0,0.0,20.372662,0.0,0.0,1.0,0.635211,0.263042,0.327848,0.059064,-0.047370,0.021186,1.221700,0.528200,0.505219,-1.027104,0.031012,-0.006429,-0.052606,0.332243,3,3 - Elevated risk / leveraged
3,Non-financial,3,5529,1733,58.333333,0.034464,0.000000,1.000000,1.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.0,1.000000,0.0,0.0,18.896822,0.0,0.0,0.0,0.299106,0.118076,0.603928,0.265837,-0.254194,-0.193932,4.468526,2.129241,0.229954,-12.549167,-0.516865,-1.156959,-0.401186,0.417091,4,4 - High risk / speculative
4,Non-financial,4,2092,1108,71.625498,0.718471,0.537520,1.000000,1.000000,1.000000,1.0,1.000000,0.319360,1.000000,0.000000,0.678052,0.575516,1.000000,1.000000,1.0,1.000000,1.0,0.0,19.083987,0.0,0.0,0.0,1.021599,0.441616,-0.153169,0.125123,-0.150641,-0.066158,1.152435,0.568363,0.469216,-3.657879,-0.094049,-0.217978,-0.180671,0.350005,5,5 - Distressed / near-default proxy
